# Orquestrador de Treino — Lesões de Pele (HAM10000)

Este notebook liga as quatro peças que vivem em `src/` (`data.py`,
`models.py`, `training.py`, `utils.py`) numa execução única: aquisição do
HAM10000 no Kaggle, montagem do inventário particionado por lesão (sem
vazamento entre imagens da mesma lesão), EDA visual/estatística, baseline
não trivial (features de cor+textura), treino da CNN de referência,
interpretação de erro, e uma variante ilustrativa LSTM-sobre-patches
("Etapa 2") — todos comparados no mesmo log de experimentos.

**Feito para rodar no Google Colab.** O código-fonte (`src/`) é obtido via
`git clone` com sparse-checkout do seu repositório. As únicas coisas que
precisam de ajuste manual estão marcadas com `# AJUSTE` nos comentários: a
URL do repositório (seção 1) e, opcionalmente, onde salvar dados/artefatos
(seção 2).

## 1. Código-fonte e dependências

Primeiro clona o repositório (para ter `src/` e o `requirements.txt`
disponíveis) e só então instala as dependências a partir desse
`requirements.txt` — evita manter a lista de pacotes duplicada entre o
notebook e o repositório.

In [ ]:
# AJUSTE: URL do seu repositório git e o caminho da subpasta deste projeto
# dentro dele (clone via sparse-checkout — só essa subpasta é baixada, o
# resto do repositório nem chega a ser transferido).
REPO_URL = ""
SUBPASTA = "projeto_2_cnn"
REPO_DIR = "/content/repo"

if not REPO_URL:
    raise ValueError("Defina REPO_URL com a URL do seu repositório git antes de continuar.")

import os
import subprocess

PROJECT_DIR = os.path.join(REPO_DIR, SUBPASTA)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # --filter=blob:none: baixa só os metadados de commit/árvore, os
    # arquivos (blobs) de fora do sparse-checkout nunca são transferidos.
    # --sparse liga o modo cone do sparse-checkout (mais rápido e mais
    # simples de configurar que listar padrões manualmente).
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--sparse", "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )
    subprocess.run(["git", "sparse-checkout", "set", SUBPASTA], cwd=REPO_DIR, check=True)
else:
    # Repositório já clonado nesta sessão do Colab (variável de estado que
    # sobrevive a reexecuções de célula, só não a reinício de runtime) —
    # git pull traz commits novos em vez de simplesmente pular, para
    # mudanças feitas no código depois do clone inicial não ficarem presas
    # do lado de fora.
    print(f"Repositório já clonado em '{REPO_DIR}' — buscando mudanças mais recentes...")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)


In [ ]:
# torch/torchvision já vêm prontos no Colab; polars, scikit-image e a CLI
# do kaggle não. pip install -r é idempotente — reinstalar uma dependência
# já satisfeita é rápido.
import sys

requirements_path = os.path.join(PROJECT_DIR, "requirements.txt")
# --upgrade: o Colab pode vir com uma versão antiga do pacote 'kaggle'
# pré-instalada, que satisfaz "kaggle>=1.6.0" sem precisar atualizar — só
# que essa versão antiga não tem suporte a algumas funcionalidades atuais.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "-r", requirements_path],
    check=True,
)


## 2. Onde salvar dados e artefatos

Este projeto trabalha com milhares de imagens pequenas — ler direto do
Drive via FUSE é lento o bastante para atrapalhar o treino. Por isso há
duas camadas de armazenamento:

- **`BASE_DIR`** (Drive, se `SALVAR_NO_DRIVE = True`): dataset bruto,
  inventário particionado, checkpoints, logs — persiste entre reinícios
  de runtime.
- **`LOCAL_DIR`** (sempre `/content`, nunca Drive): cópia local das
  imagens para leitura rápida durante o treino — efêmera por design (some
  ao reiniciar o runtime), mas barata de refazer (seção 6.2).

**AJUSTE** `SALVAR_NO_DRIVE` e o caminho de `BASE_DIR` dentro do Drive, se
necessário.

In [ ]:
# Onde salvar dados e artefatos desta execução.
SALVAR_NO_DRIVE = True  # AJUSTE conforme sua preferência.

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # AJUSTE: caminho da pasta de saída dentro do seu Drive.
    BASE_DIR = "/content/drive/MyDrive/rede_neural_projetos/projeto_2_cnn"
else:
    BASE_DIR = "/content/projeto_2_cnn_saida"

BASE_DIR = os.path.abspath(BASE_DIR)
print(f"BASE_DIR = {BASE_DIR} (Drive: {SALVAR_NO_DRIVE})")


In [ ]:
# Estrutura de saída (Drive/BASE_DIR) + cópia local das imagens (sempre /content).
DIR_DADOS_RAW = os.path.join(BASE_DIR, "data", "raw")
DIR_DADOS_PROCESSADOS = os.path.join(BASE_DIR, "data", "processed")
DIR_OUTPUTS = os.path.join(BASE_DIR, "outputs")
DIR_CHECKPOINTS = os.path.join(DIR_OUTPUTS, "checkpoints")
DIR_REPORT_ASSETS = os.path.join(DIR_OUTPUTS, "report_assets")
DIR_RUNS = os.path.join(BASE_DIR, "runs")  # logs do TensorBoard

LOCAL_DIR = "/content/local_data"  # nunca no Drive — ver seção 2 acima

for d in (DIR_DADOS_RAW, DIR_DADOS_PROCESSADOS, DIR_CHECKPOINTS, DIR_REPORT_ASSETS, DIR_RUNS):
    os.makedirs(d, exist_ok=True)

print("Diretórios prontos:")
for d in (DIR_DADOS_RAW, DIR_DADOS_PROCESSADOS, DIR_CHECKPOINTS, DIR_REPORT_ASSETS, DIR_RUNS, LOCAL_DIR):
    print(f"  {d}")


## 3. Imports do projeto (`src/`)

`PROJECT_DIR` (não `BASE_DIR`) precisa estar no `sys.path`, já que os
módulos usam imports absolutos no estilo `from src.utils import ...` e é
em `PROJECT_DIR` que o `git clone` da seção 1 deixou `src/`.

In [ ]:
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np
import polars as pl
import torch
import torch.nn as nn

from src.data import (
    CLASSES,
    autenticar_kaggle,
    download_kaggle_dataset,
    sincronizar_para_local,
    carregar_inventario,
    montar_split_por_lesao,
    checar_vazamento_split,
    salvar_inventario_particionado,
    carregar_inventario_particionado,
    montar_transform,
    preparar_dataloaders,
    construir_matriz_features,
)
from src.models import (
    criar_cnn_padrao,
    criar_cnn_residual,
    criar_lstm_patches_padrao,
    treinar_baseline,
)
from src.training import (
    fixar_seeds,
    computar_pesos_classe,
    treinar_modelo,
    carregar_melhor_modelo,
    registrar_experimento,
    exportar_experimentos,
)
from src.utils import (
    log_etapa,
    log_nota,
    exportar_log,
    amostra_visual_por_classe,
    checar_imagens_corrompidas,
    distribuicao_dimensoes,
    estatisticas_pixel,
    plot_curvas_loss,
    plot_gradient_norm,
    avaliar_baseline,
    avaliar_cnn,
    plotar_matriz_confusao,
    exemplos_classificados_errado,
)

print("Módulos do projeto importados com sucesso.")


## 4. Reprodutibilidade e dispositivo

`fixar_seeds` precisa rodar **antes** de qualquer coisa que consuma
aleatoriedade global — instanciação dos modelos (inicialização de pesos) e
criação dos `DataLoader` de treino (`shuffle=True`).

In [ ]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# priorizar_velocidade=True: cudnn.benchmark ligado — mais rápido em GPU,
# ao custo de reprodutibilidade bit-a-bit exata (ver docstring de
# fixar_seeds em training.py). Troque para False só se precisar comparar
# execuções byte a byte.
fixar_seeds(SEED, priorizar_velocidade=True)
print(f"Dispositivo selecionado: {DEVICE}")

from datetime import datetime
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID = {RUN_ID}")


## 5. Aquisição do dataset (Kaggle — HAM10000)

Dataset: [`kmader/skin-cancer-mnist-ham10000`](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000)
— extrai direto em `DIR_DADOS_RAW` (sem subpasta intermediária): um CSV de
metadados (`HAM10000_metadata.csv`) e duas pastas de imagens
(`HAM10000_images_part_1`/`_2`).

Requer uma credencial do Kaggle (kaggle.com/settings -> API): o token novo
(`KGAT_...`) ou o `kaggle.json` legado — a célula abaixo detecta qual dos
dois você enviou pelo **conteúdo** do arquivo, não pelo nome.

In [ ]:
METADATA_CSV = os.path.join(DIR_DADOS_RAW, "HAM10000_metadata.csv")
IMAGES_DIR_1 = os.path.join(DIR_DADOS_RAW, "HAM10000_images_part_1")
IMAGES_DIR_2 = os.path.join(DIR_DADOS_RAW, "HAM10000_images_part_2")
TOKEN_PATH = os.path.join(BASE_DIR, "kaggle_config", "token.txt")

if os.path.exists(METADATA_CSV):
    print(f"CSV já presente em '{METADATA_CSV}' — nenhuma aquisição necessária.")
else:
    print("CSV ainda não encontrado — será necessário baixar do Kaggle.")


In [ ]:
if not os.path.exists(METADATA_CSV) and not os.path.exists(TOKEN_PATH):
    print("Faça upload da sua credencial do Kaggle — kaggle.com/settings -> API:")
    print("- 'Generate New Token' -> token novo (uma linha, tipo 'KGAT_...'), OU")
    print("- 'Create Legacy API Key' -> kaggle.json (antigo)")
    from google.colab import files
    uploaded = files.upload()
    _, conteudo = next(iter(uploaded.items()))

    # Detecta o formato pelo CONTEÚDO do arquivo, não pelo nome — o nome do
    # .txt do token novo pode variar dependendo de como foi baixado.
    # kaggle.json é um JSON (começa com '{'); o token novo é uma linha só,
    # e é justamente o formato que autenticar_kaggle() espera aqui.
    texto = conteudo.decode("utf-8").strip()
    if texto.startswith("{"):
        raise ValueError(
            "Este projeto espera o token novo do Kaggle (KGAT_...), não o "
            "kaggle.json legado. Gere um em kaggle.com/settings -> API -> "
            "'Generate New Token'."
        )
    os.makedirs(os.path.dirname(TOKEN_PATH), exist_ok=True)
    with open(TOKEN_PATH, "w", encoding="utf-8") as f:
        f.write(texto)
    print(f"Token salvo em '{TOKEN_PATH}'.")


In [ ]:
download_kaggle_dataset(
    dest_dir=DIR_DADOS_RAW,
    metadata_csv=METADATA_CSV,
    token_path=TOKEN_PATH,
)


## 6. Inventário, sincronização local e split por lesão

### 6.1 Sincronizar imagens para o disco local do runtime

Efêmero por design (some ao reiniciar o runtime) — se a sessão cair, só
rodar esta célula de novo.

In [ ]:
caminhos_locais = sincronizar_para_local(
    metadata_csv_origem=METADATA_CSV,
    images_dir_1_origem=IMAGES_DIR_1,
    images_dir_2_origem=IMAGES_DIR_2,
    local_dir=LOCAL_DIR,
)
caminhos_locais


### 6.2 Inventário e split por lesão

`carregar_inventario` procura cada imagem primeiro na cópia local (rápida)
— por isso a lista de pastas começa pelas locais. `montar_split_por_lesao`
garante que todas as imagens de uma mesma lesão caiam no mesmo split;
`checar_vazamento_split` confirma isso depois de construído.

In [ ]:
df = carregar_inventario(
    metadata_csv=caminhos_locais["metadata_csv"],
    pastas_imagens=[caminhos_locais["images_dir_1"], caminhos_locais["images_dir_2"], IMAGES_DIR_1, IMAGES_DIR_2],
)
df = montar_split_por_lesao(df, proporcoes=(0.70, 0.15, 0.15), seed=SEED)
vazamentos = checar_vazamento_split(df)
assert not vazamentos, f"Vazamento de lesion_id entre splits: {vazamentos}"

INVENTARIO_PATH = os.path.join(DIR_DADOS_PROCESSADOS, "inventario_particionado.parquet")
salvar_inventario_particionado(df, INVENTARIO_PATH)


## 7. EDA — inspeção visual e estatística das imagens

### 7.1 Amostra visual por classe

In [ ]:
amostra_visual_por_classe(df, coluna_classe="dx", n_por_classe=3, seed=SEED)


### 7.2 Imagens corrompidas

Varre o dataset inteiro — pode demorar alguns minutos na primeira vez,
mas é melhor descobrir agora do que no meio do treino.

In [ ]:
corrompidas = checar_imagens_corrompidas(df)


### 7.3 Distribuição de dimensões (amostra)

In [ ]:
distribuicao_dimensoes(df, amostra=500)


### 7.4 Estatísticas de pixel (média/desvio RGB)

Insumo direto para a normalização em `montar_transform`/`preparar_dataloaders`
— usa as estatísticas REAIS deste dataset, não os valores padrão do ImageNet.

In [ ]:
stats_pixel = estatisticas_pixel(df, amostra=200, seed=SEED)
MEDIA_RGB = stats_pixel["media_rgb"]
DESVIO_RGB = stats_pixel["desvio_rgb"]
stats_pixel


## 8. DataLoaders PyTorch

Resize direto para o quadrado (distorce a proporção 4:3 original — decisão
de simplicidade, documentada em `data.montar_transform`), normalizado com
`MEDIA_RGB`/`DESVIO_RGB` calculados na seção 7.4.

`aumentar_treino=True`: só o split de TREINO recebe augmentation geométrico
(flip horizontal/vertical, rotação, translação/escala leves) — lesões de
pele não têm orientação canônica, então essas transformações preservam o
rótulo. Val/test usam sempre o transform sem augmentation. Como os
`dataloaders` são construídos uma única vez aqui e reaproveitados por
TODOS os modelos treinados a seguir (CNN de referência, variante residual,
LSTM sobre patches), a comparação entre arquiteturas nas seções 10 e 11
continua isolando só a arquitetura — todas se beneficiam igualmente da
augmentation.

In [ ]:
BATCH_SIZE = 32
dataloaders = preparar_dataloaders(df, MEDIA_RGB, DESVIO_RGB, batch_size=BATCH_SIZE, tamanho=128, num_workers=2, aumentar_treino=True)

df_treino = df.filter(pl.col("split") == "train")
df_val = df.filter(pl.col("split") == "val")
df_teste = df.filter(pl.col("split") == "test")


## 9. Baseline não trivial (scikit-learn)

Em vez de pixel bruto, extrai features artesanais que correspondem a
critérios clínicos reais de avaliação de lesão de pele (histograma de cor
+ textura LBP) e treina uma regressão logística por cima — compara
"conhecimento de domínio codificado à mão" vs. "features aprendidas
automaticamente pela CNN" (seção 10), não só "simples vs. complexo".

A extração de features roda imagem a imagem em Python puro — mais lenta
que o carregamento via `DataLoader` da CNN; para o dataset completo pode
levar alguns minutos por split.

In [ ]:
X_treino_bl, y_treino_bl = construir_matriz_features(df_treino)
X_val_bl, y_val_bl = construir_matriz_features(df_val)
X_teste_bl, y_teste_bl = construir_matriz_features(df_teste)


In [ ]:
resultado_baseline = treinar_baseline(X_treino_bl, y_treino_bl, X_val_bl, y_val_bl)
metricas_baseline = avaliar_baseline(resultado_baseline, X_teste_bl, y_teste_bl, CLASSES)
plotar_matriz_confusao(metricas_baseline["matriz_confusao"], CLASSES, titulo="Baseline — Regressão Logística", salvar_dir=DIR_REPORT_ASSETS, nome_modelo="baseline")

registrar_experimento(
    nome="baseline_logistica",
    tipo="baseline",
    hiperparametros={"modelo": "LogisticRegression", "class_weight": "balanced", "features": "cor+LBP (58-d)"},
    metricas={"f1_macro": metricas_baseline["f1_macro"], "accuracy": metricas_baseline["accuracy"]},
    notas="Referência mínima — medição única, sem busca de hiperparâmetros.",
)
metricas_baseline["f1_macro"], metricas_baseline["accuracy"]


## 10. CNN de referência

Arquitetura v1: 4 blocos convolucionais (16→32→64→128 filtros), Global
Average Pooling em vez de Flatten+Dense (evita milhões de parâmetros numa
única camada densa, dado o tamanho modesto do dataset), Dropout2d espacial
dentro dos blocos + Dropout comum antes da saída — ver `models.CNNLesaoPele`.

### 10.1 Pesos de classe + treino

A seção 10.5 treina uma segunda arquitetura (`CNNLesaoPeleResidual`, com conexões de atalho e mais parâmetros) sob a MESMA infraestrutura — mesmos DataLoaders, mesma ponderação de classe, mesmos hiperparâmetros de treino — para um teste controlado, comparando só a arquitetura, não uma dúzia de fatores ao mesmo tempo.

In [ ]:
pesos_classe = computar_pesos_classe(df_treino)
modelo_cnn = criar_cnn_padrao(num_classes=len(CLASSES))

CHECKPOINT_CNN = os.path.join(DIR_CHECKPOINTS, f"cnn_{RUN_ID}")
historico_cnn = treinar_modelo(
    modelo_cnn,
    dataloaders,
    pesos_classe,
    nome_experimento=f"cnn_{RUN_ID}",
    checkpoint_dir=DIR_CHECKPOINTS,
    dir_runs=DIR_RUNS,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    paciencia_early_stopping=7,
    device=DEVICE,
    clip_grad_norm_max=5.0,
)


### 10.2 Diagnóstico de treino

In [ ]:
modelo_cnn = carregar_melhor_modelo(modelo_cnn, os.path.join(DIR_CHECKPOINTS, f"cnn_{RUN_ID}.pt"), device=DEVICE)

plot_curvas_loss(historico_cnn, "cnn", salvar_dir=DIR_REPORT_ASSETS)
plot_gradient_norm(historico_cnn, "cnn", salvar_dir=DIR_REPORT_ASSETS)


### 10.3 Avaliação final (conjunto de teste)

In [ ]:
metricas_cnn = avaliar_cnn(modelo_cnn, dataloaders["test"], CLASSES, device=DEVICE)
plotar_matriz_confusao(metricas_cnn["matriz_confusao"], CLASSES, titulo="CNN de referência", salvar_dir=DIR_REPORT_ASSETS, nome_modelo=f"cnn_{RUN_ID}")

registrar_experimento(
    nome=f"cnn_{RUN_ID}",
    tipo="cnn",
    hiperparametros={"filtros": [16, 32, 64, 128], "dropout2d": 0.1, "dropout_final": 0.3, "batch_size": BATCH_SIZE, "lr": 1e-3},
    metricas={"f1_macro": metricas_cnn["f1_macro"], "accuracy": metricas_cnn["accuracy"]},
    notas="Comparar com baseline_logistica (seção 9).",
)
metricas_cnn["f1_macro"], metricas_cnn["accuracy"]


### 10.4 Interpretação de erro

Pares (classe real, classe predita) mais frequentes de erro, com exemplos
visuais — insumo direto para discutir onde o modelo confunde lesões
clinicamente parecidas.

In [ ]:
transform_avaliacao = montar_transform(MEDIA_RGB, DESVIO_RGB, tamanho=128)
df_erros_cnn = exemplos_classificados_errado(modelo_cnn, df_teste, transform_avaliacao, CLASSES, n_por_par=3, device=DEVICE)


### 10.5 Arquitetura alternativa — CNN residual (teste controlado)

`CNNLesaoPeleResidual`: entrada com stride maior + 4 blocos convolucionais organizados em 2 estágios, cada estágio fechado por uma conexão de atalho (projeção 1x1) somada à saída — ajuda o gradiente a fluir por uma rede mais funda sem se dissipar. Também troca GAP direto pra saída (como a referência) por GAP + uma camada densa oculta (256 → 128) antes da saída — mais capacidade de decisão não-linear no topo, ao custo de mais parâmetros treináveis. Mesmos hiperparâmetros de treino da seção 10.1, de propósito — a única variável isolada aqui é a arquitetura.

In [ ]:
modelo_cnn_residual = criar_cnn_residual(num_classes=len(CLASSES))

CHECKPOINT_CNN_RESIDUAL = os.path.join(DIR_CHECKPOINTS, f"cnn_residual_{RUN_ID}")
historico_cnn_residual = treinar_modelo(
    modelo_cnn_residual,
    dataloaders,
    pesos_classe,
    nome_experimento=f"cnn_residual_{RUN_ID}",
    checkpoint_dir=DIR_CHECKPOINTS,
    dir_runs=DIR_RUNS,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    paciencia_early_stopping=7,
    device=DEVICE,
    clip_grad_norm_max=5.0,
)


#### Diagnóstico de treino (arquitetura residual)

In [ ]:
modelo_cnn_residual = carregar_melhor_modelo(modelo_cnn_residual, os.path.join(DIR_CHECKPOINTS, f"cnn_residual_{RUN_ID}.pt"), device=DEVICE)

plot_curvas_loss(historico_cnn_residual, "cnn_residual", salvar_dir=DIR_REPORT_ASSETS)
plot_gradient_norm(historico_cnn_residual, "cnn_residual", salvar_dir=DIR_REPORT_ASSETS)


#### Avaliação final (conjunto de teste) — arquitetura residual

In [ ]:
metricas_cnn_residual = avaliar_cnn(modelo_cnn_residual, dataloaders["test"], CLASSES, device=DEVICE)
plotar_matriz_confusao(metricas_cnn_residual["matriz_confusao"], CLASSES, titulo="CNN residual", salvar_dir=DIR_REPORT_ASSETS, nome_modelo=f"cnn_residual_{RUN_ID}")

n_parametros_padrao = sum(p.numel() for p in modelo_cnn.parameters())
n_parametros_residual = sum(p.numel() for p in modelo_cnn_residual.parameters())

registrar_experimento(
    nome=f"cnn_residual_{RUN_ID}",
    tipo="cnn",
    hiperparametros={"arquitetura": "residual", "dropout2d": 0.25, "batch_size": BATCH_SIZE, "lr": 1e-3, "n_parametros": n_parametros_residual},
    metricas={"f1_macro": metricas_cnn_residual["f1_macro"], "accuracy": metricas_cnn_residual["accuracy"]},
    notas=f"Teste controlado contra a CNN de referência (mesmos DataLoaders/hiperparâmetros) — {n_parametros_residual:,} parâmetros vs. {n_parametros_padrao:,} da referência.",
)
metricas_cnn_residual["f1_macro"], metricas_cnn_residual["accuracy"]


#### Comparação — CNN de referência vs. CNN residual

As duas arquiteturas foram treinadas com os MESMOS DataLoaders (augmentation inclusive), a mesma ponderação de classe e os mesmos hiperparâmetros de otimização — a única variável isolada é a arquitetura em si. Vale mais como decisão empírica do que teórica: a arquitetura residual tem ~3,5x mais parâmetros; se o F1 não melhorar proporcionalmente (ou piorar), é sinal de overfitting pela capacidade extra num dataset de só ~10 mil imagens.

In [ ]:
comparacao_arquiteturas = pl.DataFrame([
    {"modelo": "Baseline (regressão logística)", "f1_macro": metricas_baseline["f1_macro"], "accuracy": metricas_baseline["accuracy"], "n_parametros": None},
    {"modelo": "CNN de referência", "f1_macro": metricas_cnn["f1_macro"], "accuracy": metricas_cnn["accuracy"], "n_parametros": n_parametros_padrao},
    {"modelo": "CNN residual", "f1_macro": metricas_cnn_residual["f1_macro"], "accuracy": metricas_cnn_residual["accuracy"], "n_parametros": n_parametros_residual},
])
log_etapa("Comparação de arquiteturas - CNN de referência vs. CNN residual", comparacao_arquiteturas)

delta_f1_residual = metricas_cnn_residual["f1_macro"] - metricas_cnn["f1_macro"]
comentario_residual = (
    "A arquitetura residual superou a referência - a capacidade extra (conexões de atalho + camada densa oculta) ajudou."
    if delta_f1_residual > 0.01 else
    "A arquitetura residual não superou a referência de forma relevante, apesar de ter bem mais parâmetros - "
    "sinal de que a capacidade extra não estava sendo o fator limitante (ou está overfitando) neste dataset."
    if delta_f1_residual <= 0.01 and abs(delta_f1_residual) <= 0.02 else
    "A arquitetura residual teve F1 pior que a referência - provável overfitting pela capacidade extra num dataset de ~10 mil imagens."
)
log_nota(
    f"CNN de referência vs. residual: f1_macro referência={metricas_cnn['f1_macro']:.4f} "
    f"({n_parametros_padrao:,} parâmetros), residual={metricas_cnn_residual['f1_macro']:.4f} "
    f"({n_parametros_residual:,} parâmetros) — delta={delta_f1_residual:+.4f}. {comentario_residual}"
)
comparacao_arquiteturas


## 11. Variante ilustrativa — LSTM sobre patches (Etapa 2)

Implementação simplificada de Abohashish et al. (2025): a imagem é dividida
em patches, cada patch vira um vetor via uma camada linear (não uma CNN por
patch, como no artigo original — simplificação deliberada), a sequência de
patches passa por uma LSTM, e o estado oculto final alimenta o
classificador. **Teste ilustrativo**, não uma tentativa de superar a CNN da
seção 10 — ver `models.PatchSequenceLSTM`.

In [ ]:
modelo_lstm = criar_lstm_patches_padrao(num_classes=len(CLASSES), tamanho_imagem=128)

historico_lstm = treinar_modelo(
    modelo_lstm,
    dataloaders,
    pesos_classe,
    nome_experimento=f"lstm_patches_{RUN_ID}",
    checkpoint_dir=DIR_CHECKPOINTS,
    dir_runs=DIR_RUNS,
    epochs=20,
    lr=1e-3,
    weight_decay=1e-4,
    paciencia_early_stopping=5,
    device=DEVICE,
    clip_grad_norm_max=5.0,
)

modelo_lstm = carregar_melhor_modelo(modelo_lstm, os.path.join(DIR_CHECKPOINTS, f"lstm_patches_{RUN_ID}.pt"), device=DEVICE)
plot_curvas_loss(historico_lstm, "lstm_patches", salvar_dir=DIR_REPORT_ASSETS)
plot_gradient_norm(historico_lstm, "lstm_patches", salvar_dir=DIR_REPORT_ASSETS)


In [ ]:
metricas_lstm = avaliar_cnn(modelo_lstm, dataloaders["test"], CLASSES, device=DEVICE)
plotar_matriz_confusao(metricas_lstm["matriz_confusao"], CLASSES, titulo="LSTM sobre patches (Etapa 2)", salvar_dir=DIR_REPORT_ASSETS, nome_modelo=f"lstm_patches_{RUN_ID}")

registrar_experimento(
    nome=f"lstm_patches_{RUN_ID}",
    tipo="lstm_patches",
    hiperparametros={"tamanho_patch": 16, "dim_embedding": 128, "hidden_lstm": 128, "batch_size": BATCH_SIZE, "lr": 1e-3},
    metricas={"f1_macro": metricas_lstm["f1_macro"], "accuracy": metricas_lstm["accuracy"]},
    notas="Teste ilustrativo (Etapa 2) — não é uma tentativa de superar a CNN da seção 10.",
)
metricas_lstm["f1_macro"], metricas_lstm["accuracy"]


## 12. Exportação de artefatos

Grava o log markdown (todo `log_etapa`/`log_nota` acumulado desde o início
do notebook) e a tabela de experimentos (baseline vs. CNN vs. LSTM
patches), ambos dentro de `DIR_REPORT_ASSETS`.

In [ ]:
exportar_log(path=os.path.join(DIR_REPORT_ASSETS, f"log_execucao_{RUN_ID}.md"))
exportar_experimentos(
    path_json=os.path.join(DIR_REPORT_ASSETS, "experimentos.json"),
    path_md=os.path.join(DIR_REPORT_ASSETS, "log_experimentos.md"),
)


## 13. TensorBoard (opcional)

Inspeciona interativamente as curvas de loss, learning rate e norma do
gradiente da CNN e da variante LSTM, direto no notebook.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DIR_RUNS}


## Resumo desta execução

- `RUN_ID`: identifica os checkpoints (`outputs/checkpoints/`) e os runs
  do TensorBoard (`runs/`) gerados por esta rodada específica.
- `metricas_baseline` / `metricas_cnn` / `metricas_lstm`: comparação de
  três abordagens na mesma tarefa (classificação de 7 classes de lesão de
  pele), todas avaliadas no mesmo conjunto de teste.
- `df_erros_cnn`: DataFrame completo de erros da CNN (não só os pares
  plotados na seção 10.4) — útil para investigar mais a fundo.
- Log completo e tabela de experimentos exportados em
  `outputs/report_assets/`.

Para uma nova rodada de treino, volte à seção 4 para gerar um novo
`RUN_ID` antes de re-executar as seções 10/11 — assim o run anterior não é
sobrescrito. `LOCAL_DIR` (seção 6.1) precisa ser resincronizado a cada
reinício de runtime; `BASE_DIR`/dados processados (Drive) não.